[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/02_sizing_and_serving/02.1_capacity_planning/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/02_sizing_and_serving/02.1_capacity_planning/lab.ipynb)

# Lab 2.1: Capacity Planning and GPU Selection

Interactive deployment calculator.

In [1]:
import numpy as np  # Import dependency
import matplotlib.pyplot as plt  # Import dependency
import ipywidgets as widgets  # Import dependency
from IPython.display import display, HTML  # Import dependency

# Model specifications: params in billions, layer/head/dim counts
MODELS = {  # Assign value
    'Mistral-7B':  {'params_B': 7.24,  'layers': 32,  'heads': 32,  'kv_heads': 8, 'head_dim': 128},
    'Llama-70B':   {'params_B': 70.6,  'layers': 80,  'heads': 64,  'kv_heads': 8, 'head_dim': 128},
    'Llama-405B':  {'params_B': 405.0, 'layers': 126, 'heads': 128,'kv_heads': 8, 'head_dim': 128},
}

# GPU VRAM in GB
GPUS = {  # Assign value
    'T4': 16, 'A10G': 24, 'A100-80': 80, 'H100': 80, 'H200': 141,
}

# Bytes per parameter for each precision
PRECISIONS = {  # Assign value
    'FP16': 2, 'INT8': 1, 'INT4': 0.5,
}

# Approximate cost per hour (USD) for reference
GPU_COST = {  # Assign value
    'T4': 0.53, 'A10G': 1.21, 'A100-80': 3.67, 'H100': 4.25, 'H200': 5.50,
}

print('Models:', list(MODELS.keys()))  # Display output
print('GPUs:', list(GPUS.keys()))  # Display output
print('Precisions:', list(PRECISIONS.keys()))  # Display output

# --- Batch analysis (A100-80, Mistral-7B FP16) ---

# GPU and model parameters (A100 80GB)
bandwidth_gbs = 2039        # HBM bandwidth in GB/s
model_size_gb = 14.5        # Model weights in GB (7B params in fp16)
kv_per_token_mb = 2         # KV cache per token in MB

# Batch sizes to evaluate
batch_sizes = np.array([1, 2, 4, 8, 16, 32, 64, 128])  # NumPy operation

# --- Instance catalog ---

# GPU instance catalog: real cloud offerings
GPU_INSTANCES = {  # Assign value
    "A10G (24GB)": {"vram_gb": 24, "bandwidth_gbs": 600, "flops_tflops": 31.2, "cost_per_hr": 1.01},
    "L4 (24GB)": {"vram_gb": 24, "bandwidth_gbs": 300, "flops_tflops": 30.3, "cost_per_hr": 0.81},
    "A100 40GB": {"vram_gb": 40, "bandwidth_gbs": 1555, "flops_tflops": 77.9, "cost_per_hr": 3.67},
    "A100 80GB": {"vram_gb": 80, "bandwidth_gbs": 2039, "flops_tflops": 77.9, "cost_per_hr": 5.12},
    "H100 80GB": {"vram_gb": 80, "bandwidth_gbs": 3350, "flops_tflops": 267.6, "cost_per_hr": 8.10},
    "L40S (48GB)": {"vram_gb": 48, "bandwidth_gbs": 864, "flops_tflops": 91.6, "cost_per_hr": 2.40},
    "H200 (141GB)": {"vram_gb": 141, "bandwidth_gbs": 4800, "flops_tflops": 267.6, "cost_per_hr": 12.50},
    "MI300X (192GB)": {"vram_gb": 192, "bandwidth_gbs": 5300, "flops_tflops": 163.4, "cost_per_hr": 10.00},
}

# Bytes per parameter by precision
PRECISION_BYTES = {"FP16": 2, "INT8": 1, "INT4": 0.5}  # Assign value

# Model sizes in billions of parameters
MODEL_PARAMS = {"7B": 7, "13B": 13, "70B": 70, "405B": 405}  # Assign value


Models: ['Mistral-7B', 'Llama-70B', 'Llama-405B']
GPUs: ['T4', 'A10G', 'A100-80', 'H100', 'H200']
Precisions: ['FP16', 'INT8', 'INT4']


## Step-by-Step Capacity Calculator

In [2]:
def capacity_calc(model_name, gpu_name, precision, tokens_per_conversation, num_users):
    """Show every calculation step for GPU memory capacity planning."""
    m = MODELS[model_name]  # Assign value
    gpu_vram = GPUS[gpu_name]  # Assign value
    bpp = PRECISIONS[precision]  # bytes per parameter

    # Step 1: Weight memory
    weight_gb = m['params_B'] * bpp  # billions of params * bytes = GB
    print(f'Step 1: Weight Memory')  # Display output
    print(f'  params x bytes_per_param = {m["params_B"]}B x {bpp} = {weight_gb:.1f} GB')  # Display output
    print()  # Display output

    # Step 2: KV cache per token (bytes, then convert)
    # Formula: 2 (K+V) x heads x head_dim x layers x bytes_per_value
    kv_bytes_per_token = 2 * m.get('kv_heads', m['heads']) * m['head_dim'] * m['layers'] * bpp  # Assign value
    kv_mb_per_token = kv_bytes_per_token / (1024**2)  # Assign value
    print(f'Step 2: KV Cache Per Token')  # Display output
    print(f'  2 (K+V) x {m["heads"]} heads x {m["head_dim"]} dim x {m["layers"]} layers x {bpp} bytes')  # Display output
    print(f'  = {kv_bytes_per_token:,.0f} bytes = {kv_mb_per_token:.2f} MB/token')  # Display output
    print()  # Display output

    # Step 3: KV cache per user
    kv_per_user_gb = (kv_bytes_per_token * tokens_per_conversation) / (1024**3)  # Assign value
    print(f'Step 3: KV Cache Per User')  # Display output
    print(f'  {kv_mb_per_token:.2f} MB/token x {tokens_per_conversation} tokens = {kv_per_user_gb:.2f} GB/user')  # Display output
    print()  # Display output

    # Step 4: Total KV cache for all users
    total_kv_gb = kv_per_user_gb * num_users  # Assign value
    print(f'Step 4: Total KV Cache')  # Display output
    print(f'  {kv_per_user_gb:.2f} GB/user x {num_users} users = {total_kv_gb:.1f} GB')  # Display output
    print()  # Display output

    # Step 5: Total memory (weights + KV + 10% overhead)
    overhead_gb = weight_gb * 0.10  # 10% of weights for activations/framework
    total_gb = weight_gb + total_kv_gb + overhead_gb  # Assign value
    print(f'Step 5: Total Memory Needed')  # Display output
    print(f'  weights + total_kv + overhead(10%) = {weight_gb:.1f} + {total_kv_gb:.1f} + {overhead_gb:.1f} = {total_gb:.1f} GB')  # Display output
    print()  # Display output

    # Step 6: Fit check
    fits = total_gb <= gpu_vram  # Assign value
    headroom = gpu_vram - total_gb  # Assign value
    print(f'Step 6: Does it fit?')  # Display output
    print(f'  GPU VRAM = {gpu_vram} GB')  # Display output
    print(f'  Needed   = {total_gb:.1f} GB')  # Display output
    if fits:  # Condition check
        print(f'  \U00002705 FITS ({headroom:.1f} GB headroom)')  # Display output
    else:
        print(f'  \U0000274C OOM (need {-headroom:.1f} more GB)')  # Display output
    print()  # Display output

    # Stacked bar chart: weights | KV | overhead vs GPU VRAM
    fig_2, ax_2 = plt.subplots(figsize=(8, 2))  # Matplotlib setup
    ax_2.barh(['Memory'], [weight_gb], color='#dbeafe', edgecolor='#000', label='Weights')
    ax_2.barh(['Memory'], [total_kv_gb], left=[weight_gb], color='#fef3c7', edgecolor='#000', label='KV Cache')
    ax_2.barh(['Memory'], [overhead_gb], left=[weight_gb + total_kv_gb], color='#f3e8ff', edgecolor='#000', label='Overhead')
    # GPU VRAM line
    ax_2.axvline(gpu_vram, color='red', linestyle='--', linewidth=2, label=f'{gpu_name} VRAM ({gpu_vram} GB)')  # Add vertical line
    ax_2.set_xlabel('GB')  # Set axis label
    ax_2.set_title('Memory Breakdown vs GPU Capacity')  # Set title
    ax_2.legend(loc='upper right', fontsize=8)  # Add legend
    ax_2.set_xlim(0, max(total_gb, gpu_vram) * 1.1)  # Set axis limits
    plt.tight_layout()  # Adjust layout
    plt.show()  # Render figure

# Build interactive widgets
widgets.interact(  # Widget element
    capacity_calc,
    model_name=widgets.Dropdown(options=list(MODELS.keys()), value='Mistral-7B', description='Model:'),  # Widget element
    gpu_name=widgets.Dropdown(options=list(GPUS.keys()), value='A100-80', description='GPU:'),  # Widget element
    precision=widgets.Dropdown(options=list(PRECISIONS.keys()), value='FP16', description='Precision:'),  # Widget element
    tokens_per_conversation=widgets.IntSlider(min=256, max=32768, step=256, value=2048, description='Tokens/conv:',  # Widget element
                                              style={'description_width': 'initial'}),  # Assign value
    num_users=widgets.IntSlider(min=1, max=256, step=1, value=16, description='Concurrent users:',  # Widget element
                                style={'description_width': 'initial'}),  # Assign value
);

interactive(children=(Dropdown(description='Model:', options=('Mistral-7B', 'Llama-70B', 'Llama-405B'), value=…

## Interactive Instance Selector

In [3]:
# Model selection widgets
model_dd = widgets.Dropdown(options=["7B", "13B", "70B", "405B"], value="7B", description="Model:")  # Widget element
prec_dd = widgets.Dropdown(options=["FP16", "INT8", "INT4"], value="FP16", description="Precision:")  # Widget element
ctx_slider = widgets.IntSlider(value=4096, min=512, max=32768, step=512, description="Context:")  # Widget element
batch_slider = widgets.IntSlider(value=8, min=1, max=128, step=1, description="Batch Size:")  # Widget element
itl_slider = widgets.FloatSlider(value=50.0, min=5.0, max=200.0, step=5.0, description="Max ITL (ms):")  # Widget element
out = widgets.Output()  # Widget element

def evaluate_instances(model, precision, tokens_per_conversation, batch_size, max_itl_ms):
    """Check each GPU against model requirements."""
    params_b = MODEL_PARAMS[model]  # Assign value
    bytes_per_param = PRECISION_BYTES[precision]  # Assign value
    # Model weight memory in GB
    weight_gb = params_b * bytes_per_param  # Assign value
    # KV cache per token per layer: 2 * hidden * 2 bytes (FP16 KV always)
    # Approximate: hidden ~ params_b * 1024 / 7 for scaling, layers ~ params_b * 32 / 7
    hidden = int(params_b * 1024 / 7) * 7  # rough scaling
    layers = int(params_b * 32 / 7)  # Assign value
    kv_per_token_gb = 2 * hidden * 2 * layers / 1e9  # 2 matrices, 2 bytes each
    kv_total_gb = kv_per_token_gb * tokens_per_conversation * batch_size  # Assign value
    total_vram_gb = weight_gb + kv_total_gb  # Assign value
    # ITL estimate: weight_gb * 1e9 / bandwidth_bytes_per_sec * 1000 (ms)
    results = []  # Assign value
    for name, spec in GPU_INSTANCES.items():  # Iterate
        vram_ok = total_vram_gb <= spec["vram_gb"] * 0.9  # 90% usable
        # Min ITL: time to read all weights once per token
        min_itl_ms = (weight_gb * 1e3) / spec["bandwidth_gbs"]  # GB / (GB/s) = seconds, *1e3 = ms
        itl_ok = min_itl_ms <= max_itl_ms  # Assign value
        passes = vram_ok and itl_ok  # Assign value
        results.append({"name": name, "vram_ok": vram_ok, "itl_ok": itl_ok,
                        "passes": passes, "min_itl_ms": min_itl_ms,
                        "total_vram_gb": total_vram_gb, "cost": spec["cost_per_hr"]})
    return sorted(results, key=lambda x: (not x["passes"], x["cost"]))  # Return result

def on_change(_):
    out.clear_output()
    with out:
        results = evaluate_instances(model_dd.value, prec_dd.value,  # Assign value
                                     ctx_slider.value, batch_slider.value, itl_slider.value)
        # Build HTML table
        html = "<table style='border-collapse:collapse;width:100%'>"  # Assign value
        html += "<tr style='background:#f3f4f6'><th>Instance</th><th>VRAM</th><th>ITL</th><th>Cost/hr</th><th>Status</th></tr>"
        for r in results:  # Iterate
            color = "#dcfce7" if r["passes"] else "#ffe4e6"  # green pass, rose fail
            status = "\u2705 PASS" if r["passes"] else "\u274c FAIL"  # Assign value
            html += f"<tr style='background:{color}'><td>{r['name']}</td><td>{r['total_vram_gb']:.1f}/{GPU_INSTANCES[r['name']]['vram_gb']}GB</td>"  # Assign value
            html += f"<td>{r['min_itl_ms']:.1f}ms</td><td>${r['cost']:.2f}</td><td>{status}</td></tr>"  # Assign value
        html += "</table>"  # Assign value
        display(HTML(html))  # Render display

# Connect widgets
for w in [model_dd, prec_dd, ctx_slider, batch_slider, itl_slider]:  # Iterate
    w.observe(on_change, names='value')  # Register callback

display(widgets.VBox([model_dd, prec_dd, ctx_slider, batch_slider, itl_slider, out]))  # Widget element
on_change(None)  # Initial render

## GPU Selection & Recommendation
Choose model, precision, context, batch, and latency SLO. See all GPUs ranked by cost with the cheapest highlighted.

In [4]:
# ── GPU Deployment Calculator ───────────────────────────────────────────────
# Uses Output + observe pattern (no duplicate renders on change)

sel_model = widgets.Dropdown(options=list(MODELS.keys()), value="Mistral-7B", description="Model:")  # Widget element
sel_prec  = widgets.Dropdown(options=list(PRECISIONS.keys()), value="FP16", description="Precision:")  # Widget element
sel_ctx   = widgets.IntSlider(min=512, max=16384, step=512, value=2048,  # Widget element
                               description="Context:", style={"description_width": "initial"})  # Assign value
sel_batch = widgets.IntSlider(min=1, max=128, step=1, value=16,  # Widget element
                               description="Batch size:", style={"description_width": "initial"})  # Assign value
sel_itl   = widgets.FloatSlider(min=5.0, max=100.0, step=5.0, value=30.0,  # Widget element
                                 description="SLO (ms):", style={"description_width": "initial"})  # Assign value
sel_out = widgets.Output()  # Widget element


def kv_per_user_gb(model, precision_bytes, tokens):
    """KV cache in GB for one user conversation."""
    kv_bytes = 2 * model.get("kv_heads", model["heads"]) * model["head_dim"] * model["layers"] * precision_bytes  # Assign value
    return (kv_bytes * tokens) / (1024**3)  # Return result


def eval_gpu(gpu_name, weight_gb, kv_gb_per_user, batch, slo_ms):
    """Return metrics dict for one GPU option."""
    vram  = GPUS[gpu_name]  # Assign value
    bw_gbs = {"T4": 320, "A10G": 600, "A100-80": 2039, "H100": 3350, "H200": 4800}[gpu_name]  # Assign value
    cost_hr = GPU_COST[gpu_name]  # Assign value
    overhead_gb = weight_gb * 0.10  # Assign value

    fits   = (weight_gb + kv_gb_per_user * batch + overhead_gb) <= vram  # Assign value
    itl_ms = (weight_gb + batch * kv_gb_per_user) / bw_gbs * 1000  # Assign value
    tok_s  = batch / (itl_ms / 1000) if itl_ms > 0 else 0  # Assign value
    cpm    = (cost_hr / tok_s) * (1e6 / 3600) if tok_s > 0 else 999  # Assign value

    return {"name": gpu_name, "vram": vram, "fits": fits,  # Return result
            "itl_ms": itl_ms, "tok_s": tok_s, "cpm": cpm,
            "cost_hr": cost_hr, "passes": fits and itl_ms <= slo_ms}  # Assign value


def render_table(results):
    """HTML table: green = pass, rose = fail."""
    rows = ""  # Assign value
    for r in results:  # Iterate
        color  = "#dcfce7" if r["passes"] else "#ffe4e6"
        status = "PASS" if r["passes"] else ("OOM" if not r["fits"] else "OVER SLO")  # Assign value
        cpm    = f"${r['cpm']:.2f}" if r["passes"] else "—"  # Assign value
        rows += (f"<tr style='background:{color}'>"  # Assign value
                 f"<td><b>{r['name']}</b></td><td>{r['vram']} GB</td>"
                 f"<td>{r['itl_ms']:.1f}ms</td><td>{r['tok_s']:,.0f}</td>"
                 f"<td>${r['cost_hr']:.2f}</td><td>{cpm}</td>"
                 f"<td><b>{status}</b></td></tr>")
    header = ("<table style='border-collapse:collapse;width:100%;font-size:13px'>"  # Assign value
              "<tr style='background:#f3f4f6'><th>GPU</th><th>VRAM</th><th>ITL</th>"
              "<th>Tok/s</th><th>$/hr</th><th>$/M tok</th><th>Status</th></tr>")
    return header + rows + "</table>"  # Return result


def render_chart(passing, model_name, precision, batch, slo_ms):
    """Horizontal bar chart of $/M tokens for passing GPUs."""
    names  = [r["name"] for r in passing]  # Assign value
    costs  = [r["cpm"]  for r in passing]  # Assign value
    best_i = costs.index(min(costs))  # Assign value
    colors = ["#86efac" if i == best_i else "#dbeafe" for i in range(len(names))]

    fig_sel, ax_sel = plt.subplots(figsize=(9, max(2.5, len(names) * 0.7)))  # Matplotlib setup
    ax_sel.barh(names, costs, color=colors, edgecolor="#000", height=0.5)
    for i, (n, c) in enumerate(zip(names, costs)):  # Iterate
        label = f" ${c:.2f}/M | {passing[i]['tok_s']:,.0f} tok/s"  # Assign value
        if i == best_i:  # Condition check
            label += " ← CHEAPEST"  # Assign value
        ax_sel.text(c + max(costs) * 0.02, i, label, va="center", fontsize=9)  # Assign value
    ax_sel.set_xlabel("$/Million Tokens (lower = better)")  # Set axis label
    ax_sel.set_title(f"{model_name} ({precision}) | batch={batch} | SLO={slo_ms}ms")  # Set title
    ax_sel.invert_yaxis()
    ax_sel.set_xlim(0, max(costs) * 2.2)  # Set axis limits
    ax_sel.spines["top"].set_visible(False)
    ax_sel.spines["right"].set_visible(False)
    plt.tight_layout()  # Adjust layout
    plt.show()  # Render figure


def on_change(_):
    sel_out.clear_output(wait=True)  # Assign value
    with sel_out:
        model = MODELS[sel_model.value]  # Assign value
        bpp   = PRECISIONS[sel_prec.value]  # Assign value
        kv_gb = kv_per_user_gb(model, bpp, sel_ctx.value)  # Assign value

        results  = [eval_gpu(g, model["params_B"] * bpp, kv_gb, sel_batch.value, sel_itl.value)  # Assign value
                    for g in GPUS]  # Iterate
        results.sort(key=lambda r: (not r["passes"], r["cpm"]))  # Assign value

        display(HTML(render_table(results)))  # Render display

        passing = [r for r in results if r["passes"]]  # Assign value
        if passing:  # Condition check
            render_chart(passing, sel_model.value, sel_prec.value, sel_batch.value, sel_itl.value)
            best = passing[0]  # Assign value
            print(f"\nBest: {best['name']} — ${best['cpm']:.2f}/M tokens, {best['tok_s']:,.0f} tok/s")  # Display output
        else:
            print("No GPU meets these constraints. Try increasing SLO or reducing context.")  # Display output


for w in [sel_model, sel_prec, sel_ctx, sel_batch, sel_itl]:  # Iterate
    w.observe(on_change, names="value")  # Register callback

display(widgets.VBox([sel_model, sel_prec, sel_ctx, sel_batch, sel_itl, sel_out]))  # Widget element
on_change(None)


## Key Takeaway
**KV cache is the variable cost.** The model weights are fixed; KV cache grows linearly with sequence length and batch size. Your deployment decision comes down to: which GPU gives you enough VRAM and bandwidth at the lowest $/M tokens.